# Day 2 — Step 4: 시각화 [도전 과제 문제3]

03_moving_average에서 저장한 최종 데이터를 불러와  
종가(close)와 5일 이동평균선(MA5)을 차트로 시각화합니다.

---

## 차트 레이아웃

```
┌──────────────────────────────────────────────────────────────┐
│           암호화폐 종가 및 5일 이동평균선 (최근 30일)          │
├──────────────────┬──────────────────┬────────────────────────┤
│   KRW-BTC        │   KRW-ETH        │   KRW-SOL              │
│  ─ 종가(close)   │  ─ 종가(close)   │  ─ 종가(close)         │
│  -- MA5          │  -- MA5          │  -- MA5                 │
└──────────────────┴──────────────────┴────────────────────────┘
```

### 차트 읽는 법

| 상황 | 의미 |
|------|------|
| 종가가 MA5 **위** | 단기 상승 중 |
| 종가가 MA5 **아래** | 단기 하락 중 |
| MA5선이 **우상향** | 5일 평균이 오르고 있다 = 상승 추세 |
| MA5선이 **우하향** | 5일 평균이 내리고 있다 = 하락 추세 |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt          # 차트 그리기
import matplotlib.dates as mdates        # x축 날짜 포맷
import matplotlib.ticker as mticker      # y축 숫자 포맷
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

In [ ]:
# ── 03단계에서 저장한 최종 데이터 불러오기 ───────────────────────
df = pd.read_csv('ohlcv_final.csv', parse_dates=['date'])

print(f"데이터 로드 완료: {len(df)}행")
print(df.head())

---
## [도전 과제 문제3] 종가와 5일 이동평균선 시각화

In [ ]:
def prepare_plot_data(df):
    """
    시각화 전 데이터를 종목별로 분리합니다.

    반환값:
        {"KRW-BTC": DataFrame, "KRW-ETH": DataFrame, ...} 형태의 딕셔너리
    """
    df = df.copy()

    # date 컬럼을 datetime 타입으로 확인 (mdates가 datetime 타입 필요)
    df['date'] = pd.to_datetime(df['date'])

    # 종목별로 데이터 분리
    # .unique() : 중복 없이 유일한 ticker 목록 반환
    ticker_data = {}
    for ticker in df['ticker'].unique():
        ticker_df = df[df['ticker'] == ticker].sort_values('date').copy()
        ticker_data[ticker] = ticker_df

    return ticker_data

In [ ]:
def plot_close_and_ma5(ticker_data):
    """
    3종목의 종가(close)와 5일 이동평균(ma5)을 1행 3열 서브플롯으로 시각화합니다.

    매개변수:
        ticker_data : prepare_plot_data()에서 반환된 딕셔너리
    """
    # ── 1행 3열 서브플롯 생성 ─────────────────────────────────────
    # fig   : 전체 그림 객체
    # axes  : 각 서브플롯(차트 칸) 배열 [ax0, ax1, ax2]
    # figsize=(18, 5) : 전체 가로 18인치, 세로 5인치
    fig, axes = plt.subplots(nrows=1, ncols=len(ticker_data), figsize=(18, 5))

    for i, ticker in enumerate(ticker_data):
        ax  = axes[i]               # i번째 서브플롯
        tdf = ticker_data[ticker]   # 해당 종목 데이터

        # ── 종가 선 (검정 실선) ───────────────────────────────────
        ax.plot(tdf['date'], tdf['close'],
                color='black', linewidth=1.5, label='종가(close)')

        # ── MA5 선 (빨간 점선) ────────────────────────────────────
        # linestyle='--' : 점선
        # alpha=0.8      : 약간 투명하게 (종가선이 더 잘 보이도록)
        ax.plot(tdf['date'], tdf['ma5'],
                color='red', linewidth=1.2,
                linestyle='--', alpha=0.8,
                label='MA5 (5일 이동평균)')

        # ── 차트 제목 / 축 라벨 ───────────────────────────────────
        ax.set_title(ticker, fontsize=13, fontweight='bold')
        ax.set_xlabel('날짜', fontsize=10)
        if i == 0:   # 첫 번째 칸에만 y축 라벨 표시 (중복 방지)
            ax.set_ylabel('가격 (원)', fontsize=10)

        # ── x축 날짜 포맷 ─────────────────────────────────────────
        # DateFormatter('%m-%d') : '01-05' 형식으로 날짜 표시
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=6))

        # 날짜 텍스트 45도 기울이기 (겹침 방지)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

        # ── y축 숫자 포맷 ─────────────────────────────────────────
        # FuncFormatter : 숫자에 쉼표 삽입 (89250000 → 89,250,000)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
        )

        # ── 범례 / 격자 ───────────────────────────────────────────
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3, linestyle='--')

    # ── 전체 제목 및 여백 조절 ────────────────────────────────────
    fig.suptitle('암호화폐 종가 및 5일 이동평균선 (최근 30일)',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()  # 서브플롯 간 여백 자동 최적화
    plt.show()

    print("차트 출력 완료!")

In [ ]:
# ── 시각화 실행 ───────────────────────────────────────────────────
ticker_data = prepare_plot_data(df)
plot_close_and_ma5(ticker_data)

---
## Day 2 완료 체크리스트

- [ ] **Step 1** (01_data_collection): 3종목 30일치 수집 완료
- [ ] **Step 2** (02_preprocessing): `price_change`, `price_change_pct`, `high_low_diff` 추가
- [ ] **Step 3** (03_moving_average): `ma5` 추가, NaN 결측치 없음 확인
- [ ] **Step 4** (04_visualization): 종가 + MA5 서브플롯 차트 정상 출력

---

## Day 3 예고

오늘 완성한 `ohlcv_final.csv`를  
**SQLite 데이터베이스에 저장**하고, SQL로 조회하는 방법을 배웁니다!